# Overall GAN Validation on Colab A100

This notebook stages the manuscript-revision validation bundle onto Colab local disk, copies active model checkpoints from Google Drive, extracts held-out test archives locally, writes a Colab-specific validation config, runs the full validation with CUDA and `batch_size=128`, and copies final results back to Google Drive. Runtime speed depends heavily on staging inputs to `/content`; avoid running directly against mounted Drive folders.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
import zipfile
from pathlib import Path

print('Python:', sys.version)
try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch import failed:', exc)

subprocess.run(['nvidia-smi'], check=False)


In [ ]:
INSTALL_PACKAGES = True
MOUNT_GOOGLE_DRIVE = True

if INSTALL_PACKAGES:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'rasterio', 'numpy', 'pandas', 'scipy', 'matplotlib', 'tqdm'
    ])

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


Edit the archive paths below after uploading the held-out test archives to Drive. The model roots default to the paths used by the prior Colab training notebooks: legacy models in `IM3/EvalP1/codev3` and MSA-sample models in `IM3/EvalP1/revision_msa_sample`. The latter path is taken from the current notebooks in `Script/Revision/2_ModelTraining/MSASample`, where all six active MSA-sample runs write to `_MSASample` subfolders under that root.

In [ ]:
# Drive locations. Edit these if your uploaded archives or code live elsewhere.
VALIDATION_CODE_DRIVE_DIR = '/content/drive/MyDrive/IM3/EvalP1/OverallValidation'
LEGACY_MODEL_DRIVE_ROOT = '/content/drive/MyDrive/IM3/EvalP1/codev3'
MSA_MODEL_DRIVE_ROOT = '/content/drive/MyDrive/IM3/EvalP1/revision_msa_sample'
SAN_DIEGO_TEST_ARCHIVE = '/content/drive/MyDrive/IM3/EvalP1/validation_inputs/Archive_SanDiego_TestNoOverlap.zip'
CONUS_TEST_ARCHIVE = '/content/drive/MyDrive/IM3/EvalP1/validation_inputs/Archive_CONUS_M1_random_stratified_2015_test.zip'
RESULTS_DRIVE_DIR = '/content/drive/MyDrive/IM3/EvalP1/overall_validation_results_A100'

# Local Colab staging. These folders are temporary and disappear when the runtime ends.
LOCAL_ROOT = Path('/content/gan_validation')
LOCAL_CODE_DIR = LOCAL_ROOT / 'OverallValidation'
LOCAL_MODEL_ROOT = LOCAL_ROOT / 'models'
LOCAL_TEST_ROOT = LOCAL_ROOT / 'test_data'

# Runtime settings for the validation script.
BATCH_SIZE = 128
DEVICE = 'cuda'
RESUME_FULL_RUN = True

ACTIVE_MODEL_FOLDERS = {
    'LALegacy': [
        '1_BF_UNETBaseline',
        '1_BH_UNETBaseline',
        '2A_BF_cGANRandomVecFixed',
        '2A_BH_cGANRandomVecFixed',
        '3_BF_cGANMultiRandomDiversity',
        '3_BH_cGANMultiRandomDiversity',
    ],
    'MSASample': [
        '1_BF_UNETBaseline_MSASample',
        '1_BH_UNETBaseline_MSASample',
        '2A_BF_cGANRandomVecFixed_MSASample',
        '2A_BH_cGANRandomVecFixed_MSASample',
        '3_BF_cGANMultiRandomDiversity_MSASample',
        '3_BH_cGANMultiRandomDiversity_MSASample',
    ],
}


In [ ]:
def resolve_drive_path(raw_path: str | Path) -> Path:
    path = Path(raw_path)
    candidates = [path]
    text = str(path)
    if '/MyDrive/' in text:
        candidates.append(Path(text.replace('/MyDrive/', '/My Drive/')))
    if '/My Drive/' in text:
        candidates.append(Path(text.replace('/My Drive/', '/MyDrive/')))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find Drive path. Tried: ' + ', '.join(map(str, candidates)))


def copy_file_if_needed(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        return
    shutil.copy2(src, dst)


def copy_tree_replace(src: Path, dst: Path) -> None:
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)


def copy_tree_if_missing(src: Path, dst: Path) -> None:
    if dst.exists() and any(dst.iterdir()):
        print(f'Using existing local copy: {dst}')
        return
    print(f'Copying {src} -> {dst}')
    shutil.copytree(src, dst)


def extract_archive(archive_path: Path, extract_dir: Path) -> None:
    extract_dir.mkdir(parents=True, exist_ok=True)
    suffixes = ''.join(archive_path.suffixes).lower()
    print(f'Extracting {archive_path.name} -> {extract_dir}')
    if suffixes.endswith('.zip'):
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(extract_dir)
    elif suffixes.endswith(('.tar.gz', '.tgz', '.tar')):
        with tarfile.open(archive_path) as tf:
            tf.extractall(extract_dir)
    else:
        raise ValueError(f'Unsupported archive format: {archive_path}')


def find_dataset_root(search_root: Path, required_dirs: list[str]) -> Path:
    candidates = [search_root, *[p for p in search_root.rglob('*') if p.is_dir()]]
    for candidate in candidates:
        if all((candidate / name).is_dir() for name in required_dirs):
            return candidate
    raise FileNotFoundError(f'No dataset root under {search_root} contains {required_dirs}')


def stage_archive_to_local(archive_drive_path: str, local_name: str, required_dirs: list[str]) -> Path:
    drive_archive = resolve_drive_path(archive_drive_path)
    local_archive = LOCAL_ROOT / 'archives' / drive_archive.name
    copy_file_if_needed(drive_archive, local_archive)
    extract_dir = LOCAL_TEST_ROOT / local_name
    if not extract_dir.exists() or not any(extract_dir.iterdir()):
        extract_archive(local_archive, extract_dir)
    root = find_dataset_root(extract_dir, required_dirs)
    print(f'{local_name} dataset root: {root}')
    return root


def count_csv_rows(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open() as handle:
        return max(sum(1 for _ in handle) - 1, 0)


In [ ]:
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

# Copy only the validation source files, not any previous results.
code_drive_dir = resolve_drive_path(VALIDATION_CODE_DRIVE_DIR)
LOCAL_CODE_DIR.mkdir(parents=True, exist_ok=True)
for name in ['run_overall_validation.py', 'validation_core.py', 'validation_config.json', 'README.md', 'environment.yml']:
    copy_file_if_needed(code_drive_dir / name, LOCAL_CODE_DIR / name)

# Copy only active model folders needed by the manuscript-facing comparison.
legacy_drive_root = resolve_drive_path(LEGACY_MODEL_DRIVE_ROOT)
msa_drive_root = resolve_drive_path(MSA_MODEL_DRIVE_ROOT)
local_legacy_root = LOCAL_MODEL_ROOT / 'LALegacyModels'
local_msa_root = LOCAL_MODEL_ROOT / 'MSASampleModels'
for folder in ACTIVE_MODEL_FOLDERS['LALegacy']:
    copy_tree_if_missing(legacy_drive_root / folder, local_legacy_root / folder)
for folder in ACTIVE_MODEL_FOLDERS['MSASample']:
    copy_tree_if_missing(msa_drive_root / folder, local_msa_root / folder)

# Extract held-out test data archives to local disk and infer their actual roots.
san_diego_root = stage_archive_to_local(
    SAN_DIEGO_TEST_ARCHIVE,
    'SanDiegoTestNoOverlap',
    ['central', 'Frac', 'Target'],
)
conus_root = stage_archive_to_local(
    CONUS_TEST_ARCHIVE,
    'CONUSStratifiedTest',
    ['central', 'BFrac', 'BHeight'],
)


In [ ]:
base_config_path = LOCAL_CODE_DIR / 'validation_config.json'
config = json.loads(base_config_path.read_text())
config['results_root'] = str(LOCAL_CODE_DIR / 'results')
config['device'] = DEVICE
config['batch_size'] = BATCH_SIZE
config['cache_mode'] = 'lazy'
config['model_roots'] = {
    'LALegacy': str(local_legacy_root),
    'MSASample': str(local_msa_root),
}
config['dataset_specs']['SanDiegoTestNoOverlap']['root'] = str(san_diego_root)
config['dataset_specs']['SanDiegoTestNoOverlap']['manifest'] = str(san_diego_root / 'dataset_manifest.json')
config['dataset_specs']['CONUSStratifiedTest']['root'] = str(conus_root)
config['dataset_specs']['CONUSStratifiedTest']['manifest'] = str(conus_root / 'dataset_manifest.json')

colab_config_path = LOCAL_CODE_DIR / 'validation_config_colab.json'
colab_config_path.write_text(json.dumps(config, indent=2) + '\n')
print(colab_config_path)
print(colab_config_path.read_text())


In [ ]:
cmd = [
    sys.executable,
    str(LOCAL_CODE_DIR / 'run_overall_validation.py'),
    'all',
    '--config',
    str(colab_config_path),
]
if RESUME_FULL_RUN:
    cmd.append('--resume')

def print_progress() -> None:
    results_root = LOCAL_CODE_DIR / 'results'
    point_path = results_root / 'metrics' / 'overall_point_metrics.csv'
    div_path = results_root / 'diversity' / 'overall_diversity_metrics.csv'
    log_path = results_root / 'logs' / 'completed_rows.csv'
    print(
        time.strftime('%Y-%m-%d %H:%M:%S'),
        'point', f'{count_csv_rows(point_path)}/576',
        'diversity', f'{count_csv_rows(div_path)}/576',
        'log', f'{count_csv_rows(log_path)}/1152',
    )
    if log_path.exists():
        lines = log_path.read_text().splitlines()
        if len(lines) > 1:
            print('last:', lines[-1])


print('Running:', ' '.join(cmd))
start = time.time()
process = subprocess.Popen(cmd, cwd=str(LOCAL_CODE_DIR))
while process.poll() is None:
    print_progress()
    time.sleep(300)
return_code = process.wait()
print_progress()
elapsed_hours = (time.time() - start) / 3600
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)
print(f'Validation complete in {elapsed_hours:.2f} hours')


In [ ]:
# Run this in a separate cell while the validation cell is active by interrupting only if needed,
# or run after completion to inspect final counts.
results_root = LOCAL_CODE_DIR / 'results'
point_path = results_root / 'metrics' / 'overall_point_metrics.csv'
div_path = results_root / 'diversity' / 'overall_diversity_metrics.csv'
log_path = results_root / 'logs' / 'completed_rows.csv'
print('point rows:', count_csv_rows(point_path), '/ 576')
print('diversity rows:', count_csv_rows(div_path), '/ 576')
print('log rows:', count_csv_rows(log_path), '/ 1152')
if log_path.exists():
    lines = log_path.read_text().splitlines()
    print('last log row:', lines[-1] if len(lines) > 1 else 'none')


In [ ]:
results_src = LOCAL_CODE_DIR / 'results'
results_dst = Path(RESULTS_DRIVE_DIR)
results_dst.parent.mkdir(parents=True, exist_ok=True)
if not results_src.exists():
    raise FileNotFoundError(f'Missing local results: {results_src}')
if results_dst.exists():
    shutil.rmtree(results_dst)
print(f'Copying results to Drive: {results_dst}')
shutil.copytree(results_src, results_dst)
print('Results copied to Drive.')
